In [5]:
import os
import re
import glob
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/sgd/"

# ── Derived input paths ───────────────────────────────────────────────────────

NCBI_YEAST_PATH = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Saccharomyces_cerevisiae.gene_info")


# Per-relation intermediate CSVs are read from this folder
# PROCESSED_TARKG    = os.path.join(BASE_PATH, "sgd//")

os.makedirs(OUT_PATH, exist_ok=True)

# Standard schema column order
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type", "Source",
    "KG_Source", "Head_Alt_ID", "Head_detail_name", "Tail_DO_Alt_ids",
    "Tail_IUPAC", "Tail_Smiles", "Tail_SMILES_name", "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS", "Kg_index", "Kg", "Change"
]

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/sgd/


In [7]:
# NCBI Gene

NCBI_Yeast_gene = pd.read_csv(NCBI_YEAST_PATH,sep = '\t')
NCBI_Yeast_gene

NCBI_Yeast_gene_GeneSym_2_Locus_dict = dict(zip(NCBI_Yeast_gene['Symbol'], NCBI_Yeast_gene['LocusTag']))
NCBI_Yeast_gene_GeneSym_2_Locus_dict
NCBI_Yeast_Locustag_Descrip_dict = dict(zip(NCBI_Yeast_gene['LocusTag'], NCBI_Yeast_gene['description']))
NCBI_Yeast_Locustag_Descrip_dict

{'-': 'Record to support submission of GeneRIFs for a gene not in Gene.',
 'ACI60_gp06': 'Atp6',
 'ACI60_gt03': 'tRNA-Glu',
 'ACI60_gt04': 'tRNA-Ser',
 'ACI60_gr01': 'large subunit ribosomal RNA',
 'ACI60_gt21': 'tRNA-Phe',
 'ACI60_gt24': 'tRNA-Met',
 'ACI60_gp04': 'Atp9',
 'ACI60_gp07': 'Atp8',
 'ACI60_gr02': 'small subunit ribosomal RNA',
 'ACI60_gt02': 'tRNA-Trp',
 'ACI60_gp08': 'cytochrome c oxidase subunit 1',
 'ACI60_gp05': 'cytochrome b',
 'ACI60_gp03': 'ribosomal protein S3',
 'ACI60_gt05': 'tRNA-Thr',
 'ACI60_gt06': 'tRNA-Cys',
 'ACI60_gt07': 'tRNA-His',
 'ACI60_gt08': 'tRNA-Leu',
 'ACI60_gt09': 'tRNA-Gln',
 'ACI60_gt10': 'tRNA-Lys',
 'ACI60_gt11': 'tRNA-Arg',
 'ACI60_gt12': 'tRNA-Gly',
 'ACI60_gt13': 'tRNA-Asp',
 'ACI60_gt14': 'tRNA-Ser',
 'ACI60_gt15': 'tRNA-Arg',
 'ACI60_gt16': 'tRNA-Ala',
 'ACI60_gt17': 'tRNA-Ile',
 'ACI60_gt18': 'tRNA-Tyr',
 'ACI60_gt19': 'tRNA-Asn',
 'ACI60_gt20': 'tRNA-Met',
 'ACI60_gp02': 'cytchrome c oxidase sububnit 2',
 'ACI60_gt22': 'tRNA-Thr',
 'A

In [9]:
## SGD

SGD_Gene = pd.read_csv(f'{BASE_PATH}sgd/gene_literature.tab',sep = '\t', header=None)
SGD_Gene
SGD_Gene[5] = SGD_Gene[5].str.replace('SGD:', '', regex=False)
SGD_Gene
SGD_Gene_dict = dict(zip(SGD_Gene[5],SGD_Gene[3]))
SGD_Gene_dict

SGD_Gene_dict = dict(zip(SGD_Gene[5],SGD_Gene[3]))
SGD_Gene_dict

{'S000002457': 'YDR050C',
 'S000003588': 'YJL052W',
 'S000003424': 'YGR192C',
 'S000003769': 'YJR009C',
 'S000006012': 'YPL091W',
 'S000001949': 'YFR053C',
 'S000007260': 'Q0045',
 'S000003159': 'YGL191W',
 'S000004869': 'YMR256C',
 'S000001093': 'YHR051W',
 'S000001373': 'YIL111W',
 'S000003155': 'YGL187C',
 'S000007283': 'Q0275',
 'S000007281': 'Q0250',
 'S000004387': 'YLR395C',
 'S000004028': 'YLR038C',
 'S000002225': 'YDL067C',
 'S000004997': 'YNL052W',
 'S000004860': 'YMR246W',
 'S000000817': 'YER015W',
 'S000005844': 'YOR317W',
 'S000005299': 'YNR016C',
 'S000001271': 'YIL009W',
 'S000000786': 'YEL060C',
 'S000003486': 'YGR254W',
 'S000001217': 'YHR174W',
 'S000001222': 'YHR179W',
 'S000006092': 'YPL171C',
 'S000001008': 'YHL016C',
 'S000000058': 'YAL062W',
 'S000005902': 'YOR375C',
 'S000005185': 'YNL241C',
 'S000004336': 'YLR344W',
 'S000001183': 'YHR141C',
 'S000006002': 'YPL081W',
 'S000002135': 'YER056C-A',
 'S000000933': 'YER131W',
 'S000003350': 'YGR118W',
 'S000002343': '

In [10]:
file = pd.read_csv(f'{BASE_PATH}sgd/phenotype_data.tab',sep = '\t')
file = file.drop_duplicates()
file['Head'] = file ['SGDID'].map(SGD_Gene_dict)
file = file.rename(columns={'Phenotype':'Tail'})
file['Combined_phenotype'] = file['Experiment']+'_'+file['Tail']
file['Combined_phenotype2'] = file['Tail']+'_'+file['Chemical']+'_'+file['Condition']

file['Head_type'] = 'Gene'
file['Tail_type'] = 'Phenotype'
file['tail_detail_name'] = file['Combined_phenotype']
file['Relation'] ='Gene-Phenotype'#'Gene_Aging_Phenotype'
file['kg_source'] = 'SGD'

file['head_detail_name'] = file['Head'].map(NCBI_Yeast_Locustag_Descrip_dict)

desired_order = ['Head', 'Relation', 'Tail','Head_type','Tail_type','head_detail_name', 'tail_detail_name','kg_source','Combined_phenotype','Combined_phenotype2']
file = file[desired_order]
file = file.drop_duplicates()  
# file = file[file['Feature Name']]
file

file = file[~file['head_detail_name'].isna()]
file['head_id_is'] = 'SGD_SysematicName'
file['tail_id_is'] = ''
file['relation_type'] = ''
file['species'] = 'S.cerevisae'
file
file.columns = file.columns.str.lower()
file

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,kg_source,combined_phenotype,combined_phenotype2,head_id_is,tail_id_is,relation_type,species
15,YNCN0013W,Gene-Phenotype,cold sensitivity: increased,Gene,Phenotype,ncRNA,classical genetics_cold sensitivity: increased,SGD,classical genetics_cold sensitivity: increased,cold sensitivity: increased_ _reduced temperat...,SGD_SysematicName,,,S.cerevisae
16,YNCN0013W,Gene-Phenotype,heat sensitivity: increased,Gene,Phenotype,ncRNA,classical genetics_heat sensitivity: increased,SGD,classical genetics_heat sensitivity: increased,heat sensitivity: increased_ _elevated tempera...,SGD_SysematicName,,,S.cerevisae
17,YNCN0013W,Gene-Phenotype,inviable,Gene,Phenotype,ncRNA,classical genetics_inviable,SGD,classical genetics_inviable,inviable_ _,SGD_SysematicName,,,S.cerevisae
18,YNCN0013W,Gene-Phenotype,RNA accumulation: increased,Gene,Phenotype,ncRNA,classical genetics_RNA accumulation: increased,SGD,classical genetics_RNA accumulation: increased,RNA accumulation: increased_ _permissive tempe...,SGD_SysematicName,,,S.cerevisae
26,Q0045,Gene-Phenotype,petite,Gene,Phenotype,cytochrome c oxidase subunit 1,classical genetics_petite,SGD,classical genetics_petite,petite_ _,SGD_SysematicName,,,S.cerevisae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146123,YNCM0006W,Gene-Phenotype,vegetative growth: decreased rate,Gene,Phenotype,tRNA-Tyr,systematic mutation set_vegetative growth: dec...,SGD,systematic mutation set_vegetative growth: dec...,vegetative growth: decreased rate_galactose (1...,SGD_SysematicName,,,S.cerevisae
146124,YNCM0006W,Gene-Phenotype,viable,Gene,Phenotype,tRNA-Tyr,systematic mutation set_viable,SGD,systematic mutation set_viable,viable_ _,SGD_SysematicName,,,S.cerevisae
146125,YNCM0037W,Gene-Phenotype,viable,Gene,Phenotype,tRNA-Tyr,systematic mutation set_viable,SGD,systematic mutation set_viable,viable_ _,SGD_SysematicName,,,S.cerevisae
146126,YNCM0037W,Gene-Phenotype,vegetative growth: decreased rate,Gene,Phenotype,tRNA-Tyr,systematic mutation set_vegetative growth: dec...,SGD,systematic mutation set_vegetative growth: dec...,vegetative growth: decreased rate_sodium chlor...,SGD_SysematicName,,,S.cerevisae


In [11]:
final_df = file

set(final_df['species']), set(final_df['relation']),set(final_df['head_type']),set(final_df['tail_type']),set(final_df['kg_source']),set(final_df['head_id_is']),set(final_df['tail_id_is'])

# set(final_df['relation_type'])

# Count true NaN values
true_nan_count = final_df.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

# First, concatenate all dataframes as before (assuming `final_df` already exists from earlier step)

# Group by the 3 key columns
group_cols = ['head', 'relation', 'tail']

# For kg_source, combine with '::'
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

# Merge duplicates
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})
final_df_dedup


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,Q0045,Gene-Phenotype,hydrostatic pressure resistance: decreased,Gene,,Phenotype,SGD,SGD_SysematicName,,cytochrome c oxidase subunit 1,classical genetics_hydrostatic pressure resist...,S.cerevisae
1,Q0045,Gene-Phenotype,petite,Gene,,Phenotype,SGD,SGD_SysematicName,,cytochrome c oxidase subunit 1,classical genetics_petite,S.cerevisae
2,Q0045,Gene-Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,SGD_SysematicName,,cytochrome c oxidase subunit 1,classical genetics_respiratory growth: absent,S.cerevisae
3,Q0045,Gene-Phenotype,viable,Gene,,Phenotype,SGD,SGD_SysematicName,,cytochrome c oxidase subunit 1,classical genetics_viable,S.cerevisae
4,Q0080,Gene-Phenotype,respiratory growth: absent,Gene,,Phenotype,SGD,SGD_SysematicName,,F1F0 ATP synthase subunit 8,classical genetics_respiratory growth: absent,S.cerevisae
...,...,...,...,...,...,...,...,...,...,...,...,...
73392,YPR201W,Gene-Phenotype,stress resistance: decreased,Gene,,Phenotype,SGD,SGD_SysematicName,,Arr3p,"homozygous diploid, competitive growth_stress ...",S.cerevisae
73393,YPR201W,Gene-Phenotype,utilization of nitrogen source: decreased rate,Gene,,Phenotype,SGD,SGD_SysematicName,,Arr3p,systematic mutation set (prototrophic version ...,S.cerevisae
73394,YPR201W,Gene-Phenotype,vacuolar morphology: abnormal,Gene,,Phenotype,SGD,SGD_SysematicName,,Arr3p,systematic mutation set (screen all non-essent...,S.cerevisae
73395,YPR201W,Gene-Phenotype,vegetative growth: decreased rate,Gene,,Phenotype,SGD,SGD_SysematicName,,Arr3p,large-scale survey_vegetative growth: decrease...,S.cerevisae


In [12]:
final_df_dedup['relation'] ='Gene_Phenotype'#'Gene_Aging_Phenotype'
final_df_dedup

final_df_dedup.to_csv(f'{OUT_PATH}ALL_YEAST_GENE_PHENOTYPE.csv',index = None)